
# 🧩 Mini-proyecto integrador de Data Science — SOLUCIONES



## 1. Librerías y datos sintéticos


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2025)

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

N = 400

metros = rng.normal(80, 25, size=N).clip(30, 160)
habitaciones = np.clip((metros / 30 + rng.normal(0, 0.7, size=N)).round().astype(int), 1, 6)
centro = rng.integers(0, 2, size=N)  # 0 o 1

precio = (
    15 * metros +
    50 * habitaciones +
    400 * centro +
    rng.normal(0, 200, size=N)
)

data = pd.DataFrame({
    "metros": metros,
    "habitaciones": habitaciones,
    "centro": centro,
    "precio": precio
})

data.head()



### Ejercicio 1 — Solución


In [ ]:

print("Resumen numérico:")
print(data.describe())

print("\nConteo de pisos según centro:")
print(data["centro"].value_counts())



## 3. Visualización básica


In [ ]:

plt.figure()
plt.scatter(data["metros"], data["precio"])
plt.title("Precio vs Metros")
plt.xlabel("metros"); plt.ylabel("precio (€)")
plt.show()

plt.figure()
plt.scatter(data["habitaciones"], data["precio"])
plt.title("Precio vs Habitaciones")
plt.xlabel("habitaciones"); plt.ylabel("precio (€)")
plt.show()


In [ ]:

plt.figure()
plt.hist(data.loc[data["centro"] == 1, "precio"], bins=20)
plt.title("Distribución de precios (centro=1)")
plt.xlabel("precio (€)"); plt.ylabel("frecuencia")
plt.show()

plt.figure()
plt.hist(data.loc[data["centro"] == 0, "precio"], bins=20)
plt.title("Distribución de precios (centro=0)")
plt.xlabel("precio (€)"); plt.ylabel("frecuencia")
plt.show()



### Ejercicio 2 — Solución ejemplo (texto)


In [ ]:

print("Conclusión 1: A mayor número de metros, en general el precio tiende a subir.")
print("Conclusión 2: Más habitaciones suelen asociarse con precios más altos.")
print("Conclusión 3: Los pisos con centro=1 tienen, de media, una distribución de precios desplazada hacia valores más altos.")



## 4. Matriz de diseño X y vector y


In [ ]:

X = np.c_[
    np.ones(len(data)),
    data["metros"].values,
    data["habitaciones"].values,
    data["centro"].values
]

y = data["precio"].values.reshape(-1, 1)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)



### Ejercicio 3 — Solución


In [ ]:

print("Primera fila de data:\n", data.iloc[0])
print("\nPrimera fila de X:", X[0])
print("Primer valor de y:", y[0][0])



## 5. Ajuste del modelo con OLS


In [ ]:

w = np.linalg.lstsq(X, y, rcond=None)[0]

b_hat, a1_hat, a2_hat, a3_hat = w.ravel()
print("Parámetros estimados:")
print(f"b (intercepto) ≈ {b_hat:.1f}")
print(f"a1 (metros)    ≈ {a1_hat:.1f}")
print(f"a2 (habitaciones) ≈ {a2_hat:.1f}")
print(f"a3 (centro)    ≈ {a3_hat:.1f}")

print("\nModelo estimado:")
print(f"precio ≈ {b_hat:.1f} + {a1_hat:.1f}*metros + {a2_hat:.1f}*habitaciones + {a3_hat:.1f}*centro")



### Ejercicio 4 — Comentario de solución

- `a1` indica aproximadamente cuántos euros aumenta el precio por cada metro extra.  
- `a2` indica el incremento asociado a una habitación más (a igualdad de metros).  
- `a3` indica el sobreprecio medio de estar en el centro vs no estarlo.



## 6. Evaluación: MSE y R²


In [ ]:

y_hat = (X @ w).ravel()
y_true = y.ravel()

mse = np.mean((y_true - y_hat)**2)
ss_tot = np.sum((y_true - y_true.mean())**2)
ss_res = np.sum((y_true - y_hat)**2)
r2 = 1 - ss_res/ss_tot

print("MSE del modelo:", round(mse, 2))
print("R² aproximado:", round(r2, 3))



### Ejercicio 5 — Comentario de solución

Si, por ejemplo, R² es cercano a 0.9, podemos decir que el modelo explica alrededor del 90% de la variabilidad del precio,
lo cual es bastante alto para un modelo tan simple.



## 7. Predicciones para nuevos pisos


In [ ]:

nuevos = np.array([
    [1, 70, 2, 0],
    [1, 90, 3, 1],
    [1, 50, 1, 0]
])

pred_nuevos = (nuevos @ w).ravel()

for i, precio_est in enumerate(pred_nuevos, start=1):
    print(f"Piso {i}: precio estimado ≈ {precio_est:.2f} €")



### Ejercicio 6 — Solución ejemplo


In [ ]:

nuevos_extra = np.array([
    [1, 40, 1, 0],  # pequeño, fuera del centro
    [1, 110, 4, 1], # grande, en el centro
    [1, 75, 3, 0]   # tamaño medio, fuera del centro
])

pred_extra = (nuevos_extra @ w).ravel()

for i, precio_est in enumerate(pred_extra, start=1):
    print(f"Piso extra {i}: precio estimado ≈ {precio_est:.2f} €")



## 8. Resumen final — Ejemplo de respuesta

1. Los modelos lineales combinan las variables de entrada mediante una suma ponderada, lo que permite interpretar el efecto de cada variable.  
2. El modelo es limitado porque asume una relación lineal y no captura efectos más complejos (barrio, estado del piso, planta, etc.).  
3. Para un caso real añadiría variables como: barrio, año de construcción, si tiene ascensor, si está reformado, si incluye garaje, etc.
